# ReviewMind — End-to-End Demo

This notebook walks through the full ReviewMind pipeline on a single code diff,
showing all four inference modes side by side:

| Mode | What it does |
|------|--------------|
| **Plain** | Fine-tuned LoRA model, standard prompt |
| **CoT** | Same model with a 4-step chain-of-thought suffix |
| **RAG** | Same model augmented with retrieved coding guidelines |
| **Agent** | ReAct loop — model calls tools (CVE lookup, git history, docs) before writing the review |

All four modes share the same base model (`meta-llama/Llama-3.1-8B-Instruct`)
and the same LoRA adapter (`Malak-Israr/reviewmind-lora`), loaded once at the
top of this notebook.

## 0. Install dependencies

Skip this cell if the environment already has the packages.

In [ ]:
# %pip install transformers peft bitsandbytes accelerate \
#              sentence-transformers chromadb fastapi uvicorn \
#              evaluate rouge-score -q

## 1. Project setup

In [ ]:
import sys
import time
import textwrap
from pathlib import Path

# Make project root importable regardless of where the notebook is run from
PROJECT_ROOT = Path(".").resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

## 2. The example diff

We use a realistic Python diff from a `backend/users.py` module.
The patch improves SQL parameterisation and adds type hints, but
deliberately leaves several issues a reviewer should catch:

- Passwords are stored in **plaintext** — no hashing
- The `login` function still contains **hardcoded credentials**
- `except Exception` is overly broad (swallows all errors silently)
- No connection pooling — a new SQLite connection opens on every call
- `update_password` lacks authentication — any caller can change any user's password

In [ ]:
EXAMPLE_DIFF = """\
diff --git a/backend/users.py b/backend/users.py
index a1b2c3d..e4f5g6h 100644
--- a/backend/users.py
+++ b/backend/users.py
@@ -1,28 +1,46 @@
+import logging
 import sqlite3
-from flask import request
+from typing import Optional
+from flask import request, jsonify
 
-def get_user(db_path, user_id):
-    conn = sqlite3.connect(db_path)
-    cursor = conn.cursor()
-    query = \"SELECT * FROM users WHERE id = \" + str(user_id)
-    cursor.execute(query)
-    result = cursor.fetchone()
-    conn.close()
-    return result
+logger = logging.getLogger(__name__)
 
-def update_password(db_path, user_id, new_password):
-    conn = sqlite3.connect(db_path)
-    cursor = conn.cursor()
-    cursor.execute(f\"UPDATE users SET password='{new_password}' WHERE id={user_id}\")
-    conn.commit()
-    conn.close()
+def get_user(db_path: str, user_id: int) -> Optional[dict]:
+    try:
+        conn = sqlite3.connect(db_path)
+        cursor = conn.cursor()
+        cursor.execute(
+            \"SELECT id, username, email FROM users WHERE id = ?\", (user_id,)
+        )
+        row = cursor.fetchone()
+        return {\"id\": row[0], \"username\": row[1], \"email\": row[2]} if row else None
+    except Exception as e:
+        logger.error(f\"get_user failed: {e}\")
+        return None
+    finally:
+        conn.close()
 
-def login(username, password):
-    if username == \"admin\" and password == \"admin123\":
-        return True
-    return False
+def update_password(db_path: str, user_id: int, new_password: str) -> bool:
+    if len(new_password) < 8:
+        raise ValueError(\"Password must be at least 8 characters\")
+    try:
+        conn = sqlite3.connect(db_path)
+        cursor = conn.cursor()
+        cursor.execute(
+            \"UPDATE users SET password=? WHERE id=?\", (new_password, user_id)
+        )
+        conn.commit()
+        return cursor.rowcount > 0
+    except Exception as e:
+        logger.error(f\"update_password failed: {e}\")
+        return False
+    finally:
+        conn.close()
+
+def login(username: str, password: str) -> bool:
+    # TODO: replace with proper auth
+    return username == \"admin\" and password == \"admin123\"
"""

print(EXAMPLE_DIFF)

## 3. Load the model

We load the base model in **4-bit NF4** quantisation (≈5 GB VRAM) and then
attach the LoRA adapter.  This happens once; all four review modes share the
same in-memory model object.

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL   = "meta-llama/Llama-3.1-8B-Instruct"
ADAPTER_REPO = "Malak-Israr/reviewmind-lora"

print("Loading tokenizer …")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token    = tokenizer.eos_token
    tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading base model (4-bit NF4) …")
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=False,
)

print("Attaching LoRA adapter …")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)
model.eval()
print("Model ready.")

In [ ]:
# Shared generation helper — greedy, deterministic
def generate(prompt: str, max_new_tokens: int = 300) -> str:
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1536,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()

## 4. Plain review

A straightforward prompt: system role + user message with the diff.
No extra structure.  This is the baseline fine-tuned response.

In [ ]:
SYSTEM_PROMPT = (
    "You are a senior software engineer conducting a thorough code review. "
    "Analyse the code carefully and provide specific, actionable feedback."
)

plain_prompt = (
    f"<|system|>{SYSTEM_PROMPT}<|end|>"
    f"<|user|>Review this pull request diff:\n\n{EXAMPLE_DIFF}<|end|>"
    f"<|assistant|>"
)

t0 = time.perf_counter()
plain_review = generate(plain_prompt)
plain_latency = round(time.perf_counter() - t0, 1)

print(f"Plain review  ({len(plain_review.split())} words, {plain_latency}s)")
print("=" * 70)
print(plain_review)

## 5. Chain-of-Thought (CoT) review

We append a 4-step reasoning scaffold to the user message.  The model is
prompted to think explicitly before writing the final review.  In evaluation
this mode consistently produced the longest and most structured responses.

In [ ]:
COT_SUFFIX = (
    "\n\nThink through your review step by step:\n"
    "Step 1: Understand what the code does\n"
    "Step 2: Analyse for correctness, security, performance, and readability issues\n"
    "Step 3: Prioritise which issues are critical vs minor\n"
    "Step 4: Write the final structured review"
)

cot_prompt = (
    f"<|system|>{SYSTEM_PROMPT}<|end|>"
    f"<|user|>Review this pull request diff:\n\n{EXAMPLE_DIFF}{COT_SUFFIX}<|end|>"
    f"<|assistant|>"
)

t0 = time.perf_counter()
cot_review = generate(cot_prompt)
cot_latency = round(time.perf_counter() - t0, 1)

print(f"CoT review  ({len(cot_review.split())} words, {cot_latency}s)")
print("=" * 70)
print(cot_review)

## 6. RAG-augmented review

Before prompting the model we retrieve the most relevant coding guideline
chunks from the ChromaDB vector store (built from PEP 8, OWASP Top 10, and
the Google Python Style Guide).  The retrieved text is prepended to the user
message so the model can ground its review in authoritative standards.

If the vector store is absent (e.g. first run before `python rag/build_index.py`)
this cell gracefully falls back to the plain prompt.

In [ ]:
RAG_SYSTEM_PROMPT = (
    "You are a senior software engineer conducting a thorough code review. "
    "Use the provided coding guidelines as context when reviewing the diff. "
    "Reference relevant standards and provide specific, actionable feedback."
)

retriever = None
context_block = ""

try:
    from rag.retriever import Retriever
    retriever = Retriever()
    chunks = retriever.retrieve(EXAMPLE_DIFF, top_k=3)
    context_block = retriever.format_context(chunks)
    print(f"Retrieved {len(chunks)} guideline chunks.")
    print("-" * 60)
    print(context_block[:800], "..." if len(context_block) > 800 else "")
except (SystemExit, Exception) as exc:
    print(f"[RAG] Vector store unavailable ({exc}) — falling back to plain prompt.")

if context_block:
    user_content = (
        f"{context_block}\n\n"
        "Now review the following pull request diff using the guidelines "
        f"above where relevant:\n\n{EXAMPLE_DIFF}"
    )
    rag_prompt = (
        f"<|system|>{RAG_SYSTEM_PROMPT}<|end|>"
        f"<|user|>{user_content}<|end|>"
        f"<|assistant|>"
    )
else:
    rag_prompt = plain_prompt  # fallback

t0 = time.perf_counter()
rag_review = generate(rag_prompt)
rag_latency = round(time.perf_counter() - t0, 1)

print(f"\nRAG review  ({len(rag_review.split())} words, {rag_latency}s)")
print("=" * 70)
print(rag_review)

## 7. Agent (ReAct) review

The ReAct agent runs a reasoning loop in which the model can call four tools
before writing its final review:

| Tool | Purpose |
|------|---------|
| `file_fetcher` | Returns mock file content for broader context |
| `dependency_checker` | Looks up known CVEs for imported libraries |
| `git_history` | Returns mock commit history for a function |
| `documentation_lookup` | Queries ChromaDB for relevant coding guidelines |

The loop runs for up to 8 iterations.  At each step the model emits
`Thought → Action → Observation` triples.  When it has enough information
it emits `Final Answer: <JSON>` and the loop terminates.

In [ ]:
from agent.react_agent import run_react_loop

# Wrap the shared generate() helper to match the agent's expected signature
def agent_generate_fn(mdl, tok, prompt, max_new_tokens):
    inputs = tok(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1536,
    )
    inputs = {k: v.to(mdl.device) for k, v in inputs.items()}
    with torch.no_grad():
        out_ids = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tok.eos_token_id,
        )
    new_ids = out_ids[0][inputs["input_ids"].shape[1]:]
    return tok.decode(new_ids, skip_special_tokens=True).strip()

print("Running ReAct agent (up to 8 iterations) …")
t0 = time.perf_counter()
agent_result = run_react_loop(
    diff=EXAMPLE_DIFF,
    model=model,
    tokenizer=tokenizer,
    max_iterations=8,
    max_new_tokens=400,
    generate_fn=agent_generate_fn,
)
agent_latency = round(time.perf_counter() - t0, 1)

print(f"\nAgent finished in {agent_result['n_steps']} tool call(s), {agent_latency}s")
print("\nTrace:")
for step in agent_result["trace"]:
    if step["type"] == "action":
        print(f"  [{step['step']}] {step['tool']}({step['args']})")
        print(f"       → {str(step['observation'])[:120]}")
    elif step["type"] == "final_answer":
        print(f"  [final] {str(step['content'])[:120]} …")

In [ ]:
import json

final_review = agent_result["final_review"]
print("Agent final review:")
print("=" * 70)
print(json.dumps(final_review, indent=2))

## 8. Side-by-side comparison

We collect all four reviews and print them in a unified table alongside
heuristic scores from `evaluation/score_cot.py`.

In [ ]:
from evaluation.score_cot import score_response, total as score_total

# Flatten agent review to a single string for scoring
def _flatten_agent_review(r: dict) -> str:
    parts = []
    if isinstance(r, dict):
        for v in r.values():
            if isinstance(v, str):
                parts.append(v)
            elif isinstance(v, list):
                parts.extend(str(x) for x in v)
    return " ".join(parts)

agent_text = _flatten_agent_review(final_review)

reviews = {
    "Plain":  plain_review,
    "CoT":    cot_review,
    "RAG":    rag_review,
    "Agent":  agent_text,
}
latencies = {
    "Plain":  plain_latency,
    "CoT":    cot_latency,
    "RAG":    rag_latency,
    "Agent":  agent_latency,
}

W = 80
print("=" * W)
print("  SIDE-BY-SIDE COMPARISON")
print("=" * W)
print(f"  {'Mode':<10} {'Words':>6}  {'Spec':>5}  {'Act':>5}  {'Det':>5}  {'Total':>7}  {'Latency':>9}")
print("  " + "-" * (W - 2))

for name, text in reviews.items():
    sc = score_response(text)
    tot = score_total(sc)
    words = len(text.split())
    lat = latencies[name]
    print(
        f"  {name:<10} {words:>6}  {sc['specificity']:>5}  "
        f"{sc['actionability']:>5}  {sc['detail']:>5}  {tot:>7.1f}  {lat:>8.1f}s"
    )

print("=" * W)

In [ ]:
# Full text of each review, wrapped for readability
for name, text in reviews.items():
    print()
    print(f"{'─' * 70}")
    print(f"  {name.upper()} REVIEW  ({len(text.split())} words)")
    print(f"{'─' * 70}")
    for line in text.splitlines():
        print(textwrap.fill(line, width=70, subsequent_indent="  ") if line.strip() else "")

## 9. Key findings

Based on evaluation across 20 test examples:

| Metric | Base | Fine-tuned | CoT | RAG |
|--------|------|------------|-----|-----|
| Heuristic total (/ 15) | **12.60** | 6.55 | 7.85 | 6.20 |
| ROUGE-L vs reference | 0.0548 | **0.1170** | 0.0970 | **0.1189** |
| Avg response length | 207w | 50w | 106w | 82w |

**What the two metrics reveal:**

- **Heuristic scores** (specificity, actionability, detail) reward verbosity and
  the presence of technical signals — the unmodified base model wins because it
  writes 207-word responses that naturally hit more heuristic patterns.

- **ROUGE-L** measures n-gram overlap with human-written reference reviews.
  The fine-tuned and RAG models score roughly 2× the base model here,
  confirming that fine-tuning successfully shifted the output distribution
  toward the reference style.

- **CoT is the best single augmentation**: it raises both heuristic total
  (6.55 → 7.85) and response length (50 → 106 words) while keeping ROUGE-L
  above the base model.

- **RAG adds retrieval context** at the cost of output coherence — helpful for
  standard-specific reviews but not a net gain on general diffs.

- **The agent** is the only mode that actively reasons about external
  information (CVE databases, commit history, documentation) before writing.
  Its value is highest on security-sensitive or dependency-heavy diffs like
  the one demonstrated above.